# stAI

To simplify running stAI programmatically, we modified the package so that configuration parameters can be passed directly in the code, rather than being loaded from an external YAML file.

In [ ]:
import spatialdata as sd
import scanpy as sc
import os
import pandas as pd
import stAI
import shutil
from Xenium_5K_IPF_constants import IMP_GENE_NAMES, SAMPLES_SYMBOLS, WORK_PATH, PUBLIC_PATH

# figure directory
os.makedirs(f"{WORK_PATH}Imputation/stAI", exist_ok=True)
os.makedirs(f"{WORK_PATH}Imputation/stAI/csv", exist_ok=True)

# Option
sc.settings.verbosity = 3

In [ ]:
sub_type_list = [
    "Epithelial", 
    "Macrophage", 
    "Fibroblast"
]
GSE_list = [
    "135893",
    "136831"
]
overwrite = True  # If you use data that has already been calculated using stAI, set overwrite to False.

In [ ]:
model_parameters_imp={
    'd_hidden': 64,
    'd_latent': 16,
    'lam_clf': 0.1,
    'lam_cos': 1,
    'lam_genegraph': 0,
    'lam_impute': 1,
    'lam_mmd': 0.1,
    'lam_recon': 1
}

training_parameters_imp = {
    'device_id': 0,
    'eval_step': 50,
    'k_folds': 5,
    'lr': 0.002,
    'num_epoch': 200,
    'rna_batchsize': 1024,
    'spatial_batchsize': 1024,
    'spatial_knn': 10
}

model_parameters_unm={
    'd_hidden': 64,
    'd_latent': 16,
    'lam_clf': 0.1,
    'lam_cos': 1,
    'lam_genegraph': 0,
    'lam_impute': 1,
    'lam_mmd': 0.1,
    'lam_recon': 1
}
training_parameters_unm={
    "device_id": 0,
    "k_folds": 5,
    "spatial_knn": 10
}

In [ ]:
celltype_df=pd.read_csv(f"{WORK_PATH}CSV/011_1st_clustering.csv", index_col="cell_id")

In [ ]:
for GSE in GSE_list:
    os.makedirs(f"{WORK_PATH}Imputation/stAI/GSE{GSE}", exist_ok=True)
    os.makedirs(f"{WORK_PATH}Imputation/stAI/csv/GSE{GSE}", exist_ok=True)
    
    for sub_type in sub_type_list:
        ref_adata = sc.read_h5ad(f"{PUBLIC_PATH}GSE{GSE}/ws/filtered_data.h5ad")   # filtered IPF and Control groups
        if sub_type == "Epithelial":    
            if GSE =="136831":
                adata = ref_adata[ref_adata.obs["CellType_Category"]=='Epithelial'].copy()
                adata.obs=adata.obs.rename(columns={'Subclass_Cell_Identity':"celltype"})
                adata = adata[adata.obs["celltype"]!='ATII_Low-Surfactants_SubjSpec']
                adata.obs["celltype"] = adata.obs["celltype"].cat.remove_unused_categories()
            else:  # 135893
                adata = ref_adata[ref_adata.obs["population"]=="Epithelial"].copy()
        
        elif sub_type == "Fibroblast":    
            if GSE =="136831":
                adata = ref_adata[ref_adata.obs["Subclass_Cell_Identity"].isin(["Fibroblast", 'Myofibroblast'])].copy()
                adata.obs=adata.obs.rename(columns={'Subclass_Cell_Identity':"celltype"})
                adata.obs["celltype"] = adata.obs["celltype"].cat.remove_unused_categories()
            else:  # 135893
                adata = ref_adata[ref_adata.obs["celltype"].isin(['PLIN2+ Fibroblasts', 'Fibroblasts', 'HAS1 High Fibroblasts', 'Myofibroblasts', 'Mesothelial Cells'])].copy()
        
        elif sub_type == "Macrophage":
            if GSE =="136831":
                gse_m = pd.read_table(f"{PUBLIC_PATH}GSE{GSE}/metadata.txt.gz")
                obs_df = pd.merge(left=ref_adata.obs, right=gse_m[["CellBarcode_Identity","Manuscript_Identity"]], left_index=True, right_on="CellBarcode_Identity", how="left")
                obs_df.index = obs_df["CellBarcode_Identity"]
                ref_adata.obs = obs_df[["Manuscript_Identity"]]
                adata = ref_adata[ref_adata.obs["Manuscript_Identity"].isin(['ncMonocyte', 'Macrophage_Alveolar','cMonocyte', 'Macrophage'])].copy()
                adata.obs=adata.obs.rename(columns={'Manuscript_Identity':"celltype"})
            else:  # 135893
                adata = ref_adata[ref_adata.obs["celltype"].isin(['Monocytes', 'Macrophages'])].copy()
                
        del ref_adata
        
        for i, ss in enumerate(SAMPLES_SYMBOLS):
            sp_adata = sd.SpatialData.read(f"{WORK_PATH}SpatialData/{ss}")
            df = celltype_df[celltype_df.index.str.endswith(str(i))]
            df.index = [n[:-1]+"1" for n in df.index]

            sp_adata_m = sp_adata.tables["table"]
            sp_adata_m = sp_adata_m[sp_adata_m.obs["cell_id"].isin(df.index.to_list())]
            sp_adata_m.obs = pd.merge(sp_adata_m.obs["cell_id"], df, left_on="cell_id", right_index=True) 
            sp_adata_m = sp_adata_m[sp_adata_m.obs["cell_type"]==sub_type]
        
            save_dir =f"{WORK_PATH}Imputation/stAI/GSE{GSE}/imputation_{sub_type.replace("/", "_")}_{ss}"


            if overwrite:
                if os.path.isdir(save_dir):
                    shutil.rmtree(save_dir)
                    print("Deleted old imputation data")
                print("Creat new imputation data")
                stAI.main.run_impute(
                    model_parameters=model_parameters_imp, 
                    training_parameters=training_parameters_imp,
                    adata_spatial=sp_adata_m, 
                    adata_rna=adata, 
                    save_dir=save_dir
                )
            else:
                assert os.path.isdir(save_dir), "Save directory does not exist"
                print("Use imputation data")
        
            res = stAI.main.impute_unmeasured_genes(
                genes_to_impute=IMP_GENE_NAMES[sub_type],
                adata_spatial=sp_adata_m.copy(),
                adata_rna=adata.copy(),
                training_parameters=training_parameters_unm,
                model_parameters=model_parameters_unm,
                model_dir=save_dir, 
                agg=True, 
                device="cpu"
            )
            
            df_res=pd.DataFrame(res, index=sp_adata_m.obs["cell_id"], columns=IMP_GENE_NAMES[sub_type])
                
            df_res.to_csv(f"{WORK_PATH}Imputation/stAI/csv/GSE{GSE}/{sub_type.replace("/", "_")}_imp_{ss}.csv", encoding="utf-8-sig")


# TANGRAM

In [ ]:
import spatialdata as sd
import scanpy as sc
import os
import pandas as pd
import numpy as np
import tangram as tg
import gc
import torch
from Xenium_5K_IPF_constants import SAMPLES_SYMBOLS, WORK_PATH, PUBLIC_PATH

# figure directory
os.makedirs(f"{WORK_PATH}Imputation/Tangram", exist_ok=True)

# Option
sc.settings.verbosity = 3

In [ ]:
sub_type_list = [
    "Epithelial", 
    "Macrophage", 
    "Fibroblast"
]
GSE_list = [
    "135893",
    "136831"
]

In [ ]:
celltype_df = pd.read_csv(f"{WORK_PATH}CSV/011_1st_clustering.csv")
for GSE in GSE_list:
    os.makedirs(f"{WORK_PATH}Imputation/Tangram/GSE{GSE}", exist_ok=True)
    ref_adata = sc.read_h5ad(f"{PUBLIC_PATH}GSE{GSE}/ws/filtered_data.h5ad")  # filtered IPF and Control groups
    
    if GSE == "136831":
        gse_m = pd.read_table(f"{PUBLIC_PATH}GSE{GSE}/metadata.txt.gz", index_col="CellBarcode_Identity")
        ref_adata.obs = pd.merge(ref_adata.obs, gse_m[["Manuscript_Identity"]], left_index=True, right_index=True, how="left")
    
    for sub_type in sub_type_list:
        for n, ss in enumerate(SAMPLES_SYMBOLS):
            sp_adata = sd.SpatialData.read(f"{WORK_PATH}SpatialData/{ss}")
            
            df = celltype_df[celltype_df["cell_id"].str.endswith(str(n))].copy()
            df["cell_id"] = [d[:-1]+"1" for d in df["cell_id"]]
            df.index=df["cell_id"]
            
            ad_sp = sp_adata.tables["table"]
            ad_sp = ad_sp[ad_sp.obs["cell_id"].isin(df.index.to_list())].copy()
            ad_sp.obs = pd.merge(ad_sp.obs["cell_id"],df, left_on="cell_id", right_index=True) 
            
            ad_sp = ad_sp[ad_sp.obs["cell_type"]==sub_type].copy()
                
            if sub_type == "Epithelial":    
                if GSE =="136831":
                    ad_sc = ref_adata[ref_adata.obs["CellType_Category"]=='Epithelial'].copy()
                    ad_sc.obs=ad_sc.obs.rename(columns={'Subclass_Cell_Identity':"celltype"})
                    ad_sc = ad_sc[ad_sc.obs["celltype"]!='ATII_Low-Surfactants_SubjSpec']
                    ad_sc.obs["celltype"] = ad_sc.obs["celltype"].cat.remove_unused_categories()
                else:  # 135893
                    ad_sc = ref_adata[ref_adata.obs["population"]=="Epithelial"].copy()
            
            elif sub_type == "Fibroblast":    
                if GSE =="136831":
                    ad_sc = ref_adata[ref_adata.obs["Subclass_Cell_Identity"].isin(["Fibroblast", 'Myofibroblast'])].copy()
                    ad_sc.obs=ad_sc.obs.rename(columns={'Subclass_Cell_Identity':"celltype"})
                    ad_sc.obs["celltype"] = ad_sc.obs["celltype"].cat.remove_unused_categories()
                else:  # 135893
                    ad_sc = ref_adata[ref_adata.obs["celltype"].isin(['PLIN2+ Fibroblasts', 'Fibroblasts', 'HAS1 High Fibroblasts', 'Myofibroblasts', 'Mesothelial Cells'])].copy()
            
            elif sub_type == "Macrophage":
                if GSE =="136831":
                    ad_sc = ref_adata[ref_adata.obs["Manuscript_Identity"].isin(['ncMonocyte', 'Macrophage_Alveolar','cMonocyte', 'Macrophage'])].copy()
                    ad_sc.obs=ad_sc.obs.rename(columns={'Manuscript_Identity':"celltype"})
    
                    # Sampling
                    rng = np.random.default_rng(0)
                    keep_pos = []
                    for ct, pos in ad_sc.obs.groupby("celltype").indices.items():
                        pos = np.asarray(pos)
                        k = len(pos) // 4
                        keep_pos.append(rng.choice(pos, size=k, replace=False))
                    keep_pos = np.concatenate(keep_pos) if keep_pos else np.array([], dtype=int)
                    ad_sc = ad_sc[keep_pos,].copy()
                else:  # 135893
                    ad_sc = ref_adata[ref_adata.obs["celltype"].isin(['Monocytes', 'Macrophages'])].copy()
                
            sc.pp.normalize_per_cell(ad_sp, counts_per_cell_after=1e4)
            sc.pp.log1p(ad_sp)
            sc.pp.highly_variable_genes(ad_sp, n_top_genes=2000)
            sc.pp.normalize_per_cell(ad_sc, counts_per_cell_after=1e4)
            sc.pp.log1p(ad_sc)
            sc.pp.highly_variable_genes(ad_sc, n_top_genes=4000)
            sc_ge_list = ad_sc.var[ad_sc.var["highly_variable"]].index.tolist()
            sp_ge_list = ad_sp.var[ad_sp.var["highly_variable"]].index.tolist()
            ge_list = list(set(sc_ge_list) & set(sp_ge_list))
            
            tg.pp_adatas(ad_sc, ad_sp, genes=ge_list)
            
            ad_map = tg.map_cells_to_space(ad_sc, ad_sp, device='cuda:0')
            
            ad_ge = tg.project_genes(ad_map, ad_sc)
            
            ad_ge.write(f"{WORK_PATH}Imputation/Tangram/GSE{GSE}/{ss}_{sub_type.replace("/", "_")}.h5ad")
            
            # GPU memory release
            del sp_adata, ad_sp, ad_sc, ad_map, ad_ge, df, ge_list, sc_ge_list, sp_ge_list
            gc.collect()
            torch.cuda.empty_cache()        

        adata_tangram_list = [sc.read_h5ad(f"{WORK_PATH}Imputation/Tangram/GSE{GSE}/{ss}_{sub_type.replace("/", "_")}.h5ad") for ss in SAMPLES_SYMBOLS]
            
        for n in range(len(adata_tangram_list)):
            adata_tangram_list[n].obs_names=[adata_tangram_list[n].obs["cell_id"].iloc[m][:-2]+"-"+str(n) for m in range(len(adata_tangram_list[n].obs_names))]
        adata_tangram = adata_tangram_list[0].concatenate(adata_tangram_list[1:6])
        
        adata_tangram.write(f"{WORK_PATH}Imputation/Tangram/GSE{GSE}/Tangram_{sub_type.replace("/", "_")}.h5ad")
        